We want to see if they are actually verifying the centers that have is_verified or what is happening

## 1. Imports

In [2]:
import os
import polars as pl

## 2. Load pipeline output

In [3]:
PIPELINE_DIR = os.path.join(os.path.dirname(os.path.abspath("__file__")), "initialize_vbr")
DATA_PATH = os.path.join(
    PIPELINE_DIR, "workspace", "pipelines", "initialize_vbr", "data", "quantity_data", "quantity_data_test.csv"
)

data = pl.read_csv(DATA_PATH, infer_schema_length=10000).with_columns(
    pl.col("month").cast(pl.Utf8)
)

# Keep only 2025 data
data = data.filter(pl.col("month").str.starts_with("2025"))

print(f"Loaded {len(data)} rows, {data['ou'].n_unique()} OUs, months: {sorted(data['month'].unique().to_list())}")

Loaded 304900 rows, 3477 OUs, months: ['202501', '202502', '202503', '202504', '202505', '202506', '202507', '202508', '202509', '202510', '202511', '202512']


## 3. Build the comparison table\n\nFor each OU and quarter, we count how many months have declared / verified / validated values,\nand report the `dhis2_is_not_verified` flag (already fetched by the pipeline).

In [4]:
# Per OU/month: flag presence of dec/ver/val and collapse the is_not_verified value
# (multiple rows per OU/month because each row is one service)
per_ou_month = data.group_by(["ou", "month", "quarter"]).agg(
    (pl.col("dec").cast(pl.Float64, strict=False) > 0).any().alias("has_declared"),
    (pl.col("ver").cast(pl.Float64, strict=False) > 0).any().alias("has_verified"),
    (pl.col("val").cast(pl.Float64, strict=False) > 0).any().alias("has_validated"),
    pl.col("dhis2_is_not_verified").first(),
)

# Aggregate to OU/quarter
result = (
    per_ou_month
    .group_by(["ou", "quarter"])
    .agg(
        pl.col("has_declared").sum().alias("n_months_declared"),
        pl.col("has_verified").sum().alias("n_months_verified"),
        pl.col("has_validated").sum().alias("n_months_validated"),
        pl.col("dhis2_is_not_verified").first(),
    )
    .sort(["quarter", "ou"])
)

print(f"Table shape: {result.shape}")
result

Table shape: (9144, 6)


ou,quarter,n_months_declared,n_months_verified,n_months_validated,dhis2_is_not_verified
str,str,u32,u32,u32,i64
"""A0XW9jnm1gr""","""2025Q1""",3,3,3,0
"""A2VkxpHcUvj""","""2025Q1""",3,3,3,0
"""A376kX5UkIy""","""2025Q1""",3,3,3,0
"""A3iaBulkw5X""","""2025Q1""",3,3,3,0
"""A3ifMIHaiB9""","""2025Q1""",3,3,3,0
…,…,…,…,…,…
"""znKq4Fiw8V4""","""2025Q4""",3,3,3,0
"""zsGcCqJPqoq""","""2025Q4""",3,3,3,0
"""zu21DqpDCRF""","""2025Q4""",3,3,3,0
